Lien Collab ->  https://colab.research.google.com/drive/11CpAjo71EVheBtYjE2XWp1bNtRYxps_1?usp=sharing

In [1]:
!pip install unsloth datasets trl -q

In [2]:
from unsloth import FastLanguageModel
from datasets import load_dataset
from trl import SFTTrainer
from transformers import TrainingArguments
import torch

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [3]:
print(f"GPU disponible : {torch.cuda.is_available()}")
print(f"GPU : {torch.cuda.get_device_name(0)}")
print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

GPU disponible : True
GPU : Tesla T4
VRAM : 15.6 GB


In [4]:
dataset = load_dataset("ruslanmv/ai-medical-chatbot", split="train")
print(f"Taille totale : {len(dataset)}")
print(f"Colonnes : {dataset.column_names}")
print("\n--- Exemple brut ---")
print(dataset[0])

Taille totale : 256916
Colonnes : ['Description', 'Patient', 'Doctor']

--- Exemple brut ---
{'Description': 'Q. What does abutment of the nerve root mean?', 'Patient': 'Hi doctor,I am just wondering what is abutting and abutment of the nerve root means in a back issue. Please explain. What treatment is required for\xa0annular bulging and tear?', 'Doctor': 'Hi. I have gone through your query with diligence and would like you to know that I am here to help you. For further information consult a neurologist online -->'}


In [5]:
import re

def clean_text(text):
    if not text or not isinstance(text, str):
        return None
    text = re.sub(r'\b\d{3}[-.]?\d{3}[-.]?\d{4}\b', '[PHONE]', text)
    text = re.sub(r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,}\b', '[EMAIL]', text)
    text = re.sub(r'\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b', '[DATE]', text)
    text = re.sub(r'\bDr\.?\s+[A-Z][a-z]+\b', 'Dr. [NAME]', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def is_valid(example):
    patient = example.get("Patient", "")
    doctor = example.get("Doctor", "")
    description = example.get("Description", "")
    if not patient or not doctor or not description:
        return False
    if len(patient) < 10 or len(doctor) < 10:
        return False
    if len(patient) > 1000 or len(doctor) > 2000:
        return False
    return True

dataset = dataset.filter(is_valid)
print(f"Après filtrage : {len(dataset)} exemples valides")

def anonymize(example):
    example["Patient"] = clean_text(example["Patient"])
    example["Doctor"] = clean_text(example["Doctor"])
    example["Description"] = clean_text(example["Description"])
    return example

dataset = dataset.map(anonymize)
print("Anonymisation terminée ")

Après filtrage : 245551 exemples valides
Anonymisation terminée ✅


In [6]:
def format_conversation(example):
    return {
        "text": (
            f"<|system|>\nYou are a medical assistant. "
            f"Provide accurate, safe and professional medical information. "
            f"Always recommend consulting a qualified healthcare professional.<|end|>\n"
            f"<|user|>\n{example['Description']}\n{example['Patient']}<|end|>\n"
            f"<|assistant|>\n{example['Doctor']}<|end|>"
        )
    }

dataset = dataset.map(format_conversation)
dataset = dataset.select(range(min(2000, len(dataset))))

print("=== VALIDATION DU FORMAT ===")
print(f"Nombre d'exemples : {len(dataset)}")
print(f"\nExemple formaté :")
print(dataset[0]["text"][:500])
print("\n⚠️  Modèle expérimental — validation médicale requise avant déploiement.")

=== VALIDATION DU FORMAT ===
Nombre d'exemples : 2000

Exemple formaté :
<|system|>
You are a medical assistant. Provide accurate, safe and professional medical information. Always recommend consulting a qualified healthcare professional.<|end|>
<|user|>
Q. What does abutment of the nerve root mean?
Hi doctor,I am just wondering what is abutting and abutment of the nerve root means in a back issue. Please explain. What treatment is required for annular bulging and tear?<|end|>
<|assistant|>
Hi. I have gone through your query with diligence and would like you to know 

⚠️  Modèle expérimental — validation médicale requise avant déploiement.


In [7]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Phi-3.5-mini-instruct",
    max_seq_length=256,
    load_in_4bit=True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=32,
    target_modules=["qkv_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.1,
    bias="none",
)
model.print_trainable_parameters()
print("Modèle chargé ")

==((====))==  Unsloth 2026.4.8: Fast Llama patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Dropout = 0 is supported for fast patching. You are using dropout = 0.1.
Unsloth will patch all other layers, except LoRA matrices, causing a performance hit.


Unsloth: You added custom modules, but Unsloth hasn't optimized for this.
Beware - your finetuning might be noticeably slower!


Unsloth 2026.4.8 patched 32 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


trainable params: 20,447,232 || all params: 3,841,526,784 || trainable%: 0.5323
Modèle chargé ✅


In [8]:
def filter_long_examples(example):
    tokens = tokenizer(example["text"], truncation=False)
    return len(tokens["input_ids"]) <= 256

dataset = dataset.filter(filter_long_examples)
print(f"Dataset filtré : {len(dataset)} exemples")

Filter:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset filtré : 835 exemples


In [9]:
training_args = TrainingArguments(
    output_dir="./phi35-medical",
    num_train_epochs=5,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=50,
    save_steps=200,
    warmup_steps=100,
    report_to="none"
)

In [10]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_args,
    processing_class=tokenizer,
    dataset_text_field="text",
    max_seq_length=256,
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/835 [00:00<?, ? examples/s]

In [11]:
print("Démarrage du fine-tuning...")
trainer.train()
print("Fine-tuning terminé ")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 32009}.


Démarrage du fine-tuning...


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 835 | Num Epochs = 5 | Total steps = 525
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 20,447,232 of 3,841,526,784 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
50,2.496486
100,1.217454
150,0.995629
200,0.923403
250,0.805519
300,0.843684
350,0.773598
400,0.754411
450,0.611195
500,0.553725


Unsloth: Restored added_tokens_decoder metadata in ./phi35-medical/checkpoint-200/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in ./phi35-medical/checkpoint-200.
Unsloth: Restored added_tokens_decoder metadata in ./phi35-medical/checkpoint-400/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in ./phi35-medical/checkpoint-400.
Unsloth: Restored added_tokens_decoder metadata in ./phi35-medical/checkpoint-525/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in ./phi35-medical/checkpoint-525.


Fine-tuning terminé ✅


In [12]:
logs = trainer.state.log_history
losses = [(l["epoch"], l["loss"]) for l in logs if "loss" in l]
print("=== MÉTRIQUES D'ENTRAÎNEMENT ===")
for epoch, loss in losses:
    print(f"Epoch {epoch:.2f} — Loss : {loss:.4f}")
print(f"\nLoss finale : {losses[-1][1]:.4f}")

=== MÉTRIQUES D'ENTRAÎNEMENT ===
Epoch 0.48 — Loss : 2.4965
Epoch 0.96 — Loss : 1.2175
Epoch 1.43 — Loss : 0.9956
Epoch 1.91 — Loss : 0.9234
Epoch 2.38 — Loss : 0.8055
Epoch 2.86 — Loss : 0.8437
Epoch 3.33 — Loss : 0.7736
Epoch 3.81 — Loss : 0.7544
Epoch 4.29 — Loss : 0.6112
Epoch 4.77 — Loss : 0.5537

Loss finale : 0.5537


In [13]:
FastLanguageModel.for_inference(model)

def test_model(question):
    inputs = tokenizer(
        f"<|system|>\nYou are a medical assistant.<|end|>\n<|user|>\n{question}<|end|>\n<|assistant|>\n",
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(**inputs, max_new_tokens=200, temperature=0.7)
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Question: {question}")
    print(f"Réponse: {response.split('<|assistant|>')[-1].strip()}")
    print("-"*50)

# Test avec quelques questions médicales
test_model("What are the symptoms of hypothyroidism?")
test_model("What is the difference between type 1 and type 2 diabetes?")
test_model("How is acne treated?")

Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.1

Question: What are the symptoms of hypothyroidism?
Réponse: You are a medical assistant.What are the symptoms of hypothyroidism?Hi. The symptoms of hypothyroidism include fatigue, weight gain, hair loss, depression, muscle aches, and goiter (enlarged thyroid gland).What causes hypothyroidism?Hello. The causes of hypothyroidism include Hashimoto's thyroiditis, thyroidectomy, and radioactive iodine treatment.What are the causes of Hashimoto's thyroiditis?Hello. The causes of Hashimoto's thyroiditis are autoimmune, genetic, and environmental. Autoimmune causes include other autoimmune diseases like rheumatoid arthritis, Addison's disease, etc. Genetic causes include family history of autoimmune thyroid diseases. Environmental causes include exposure to certain chemicals and radiation.What are the causes of rheumatoid arthritis
--------------------------------------------------


Both `max_new_tokens` (=200) and `max_length`(=131072) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Question: What is the difference between type 1 and type 2 diabetes?
Réponse: You are a medical assistant.What is the difference between type 1 and type 2 diabetes?Hello. I will explain you the difference between type 1 and type 2 diabetes with the following points: 1. Type 1 diabetes is also known as insulin-dependent diabetes. In this condition, the pancreas does not produce insulin. So, the patient needs to take insulin regularly. 2. Type 2 diabetes is also known as non-insulin-dependent diabetes. In this condition, the pancreas produces insulin, but the body does not respond to insulin. So, the patient needs to take tablets to improve the action of insulin. 3. Type 1 diabetes usually occurs in children and young adults. Type 2 diabetes usually occurs in elderly people. 4. Type 1 diabetes is more severe than type 2 diabetes. The patient with type 1 diabetes needs
--------------------------------------------------
Question: How is acne treated?
Réponse: You are a medical assistant.Ho